# Prompt review for QLoRA evaluation

This notebook keeps a small qualitative evaluation loop around the adapter. It prepares a repeatable prompt file and, after `qlora_lab.evaluate` is run, summarizes the generated responses for quick review.

In [ ]:
from pathlib import Path
import json

import pandas as pd

from qlora_lab.evaluate import load_prompts

prompt_dir = Path("../prompts")
report_dir = Path("../reports")
prompt_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
prompt_file = prompt_dir / "adapter_review_prompts.txt"
prompt_file.write_text(
    "\n".join(
        [
            "# Small QLoRA adapter review set",
            "Explain why gradient accumulation is useful for QLoRA.",
            "Describe the tradeoff between LoRA rank and memory use.",
            "Give a concise explanation of NF4 quantization.",
            "List two checks to run before starting a 4-bit fine-tuning job.",
        ]
    )
    + "\n"
)

prompts = load_prompts(None, prompt_file)
pd.DataFrame({"prompt": prompts, "prompt_chars": [len(prompt) for prompt in prompts]})


Run the evaluator from the project root after training an adapter:

```bash
python -m qlora_lab.evaluate \
  --adapter-dir artifacts/qlora-adapter \
  --prompt-file prompts/adapter_review_prompts.txt
```

The next cells read `reports/generations.json` if it exists.

In [ ]:
generation_path = report_dir / "generations.json"
if generation_path.exists():
    payload = json.loads(generation_path.read_text())
    generations = payload.get("generations", [])
else:
    generations = []

generation_frame = pd.DataFrame(generations)
generation_frame


In [ ]:
if not generation_frame.empty:
    review_frame = generation_frame.assign(
        prompt_chars=generation_frame["prompt"].str.len(),
        generation_chars=generation_frame["generation"].str.len(),
        generation_words=generation_frame["generation"].str.split().str.len(),
    )
else:
    review_frame = pd.DataFrame(
        columns=["prompt", "generation", "prompt_chars", "generation_chars", "generation_words"]
    )

review_frame


In [ ]:
if not review_frame.empty:
    review_frame.to_csv(report_dir / "adapter_review_summary.csv", index=False)
    review_frame[["prompt", "generation_words"]]
else:
    "No generations found yet. Run qlora_lab.evaluate after training an adapter."
